# Linear Programming Modeling: Be the Decision Maker
**IN2040 · Optimization of Organizational Processes · Tecnológico de Monterrey** (Prof. Vincent Boyer)

*Session 1 · explore during class, about 5 minutes per problem.*

> **How to use this notebook.** Run the **Setup** cell first. Then, for each problem:
> 1. read the **problem**: its data (profits, hours, audiences…) are **fixed**: they describe the situation;
> 2. play with the **explorer**: the sliders are the **decision variables**. Move them and watch the objective
>    and every constraint react (✔ satisfied, ✘ violated). Try to find the best feasible plan, then press
>    **Load the optimal plan** to compare with the solver;
> 3. read the **formulation** (the 4 steps: variables, objective, constraints, non-negativity), then run the
>    **PuLP** cell: the same model solved in Python.

For each problem, ask yourself:
1. **Which constraints stop you** when you try to improve the objective?
2. Where is the optimal plan on the graph: inside the region, on an edge, or at a **corner**?
3. How far was **your** plan from the optimum, and why?

In [ ]:
# @title ⚙️ Setup: run this cell first (no need to read the code)
import importlib, subprocess, sys
import importlib.metadata
try:                                   # PuLP 4 + its CBC solver (pulp[cbc]); replaces an older PuLP 3
    _old = int(importlib.metadata.version("pulp").split(".")[0]) < 4 or importlib.util.find_spec("cbcbox") is None
except importlib.metadata.PackageNotFoundError:
    _old = True
if _old:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pulp[cbc]>=4"], check=True)
import base64, io, json, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display
from scipy.optimize import linprog

PARTS = []      # HTML pieces of the current view, written into one HTML widget (no Output widget: VS Code safe)


def optimum(c, A_ub=None, b_ub=None, A_eq=None, b_eq=None, bounds=None, maximize=False):
    """Optimal solution of the LP (HiGHS), used for the "load the optimal plan" button."""
    c = np.asarray(c, float)
    r = linprog(-c if maximize else c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    return r.x + 0.0, (-r.fun if maximize else r.fun)


def banner(ok, text, top=False):
    color, bg = ("#067429", "#eaf9ef") if ok else ("#bd0a0a", "#fff0f0")
    (lambda h: PARTS.insert(0, h) if top else PARTS.append(h))(f'<div style="border:2px solid {color};background:{bg};color:{color};padding:8px 12px;'
                 f'border-radius:8px;font-size:15px;margin:6px 0">{text}</div>')


def table(df):
    PARTS.append(df.to_html(index=False, border=1, escape=False,
                            float_format=lambda v: f"{0.0 if abs(v) < 5e-3 else v:,.2f}")
                 .replace('<table', '<table style="border-collapse:collapse;margin:0 0 8px 0;font-size:13px"', 1))


def show(fig):
    fig.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=100); plt.close(fig)
    PARTS.append(f'<img src="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"/>')


def constraints(rows):
    """rows: (constraint, value of the left-hand side, sense, right-hand side). Shows a ✔/✘ table, returns the violated ones."""
    tol = 1e-4
    ok = [(v <= r + tol) if s == "≤" else (v >= r - tol) if s == "≥" else abs(v - r) <= tol for _, v, s, r in rows]
    table(pd.DataFrame({"Constraint": [n for n, *_ in rows], "Your plan": [v for _, v, _, _ in rows],
                        "": [s for _, _, s, _ in rows], "Limit (data)": [r for *_, r in rows],
                        "OK?": ["✔" if o else '<b style="color:#bd0a0a">✘</b>' for o in ok]}))
    return [n for (n, *_), o in zip(rows, ok) if not o]


def verdict(violated, z, z_best, what, maximize=False, unit="$"):
    """Banner: feasible or not, value of the objective, comparison with the optimum."""
    money = lambda v: f"{unit}{v:,.2f}" if unit == "$" else f"{v:,.2f} {unit}"
    if violated:
        banner(False, f"<b>INFEASIBLE plan</b>: it violates {', '.join(violated)}.<br>{what} = {money(z)}, but this "
                      f"plan is not allowed. Change the decisions until every constraint shows ✔.", top=True)
        return
    gap = (z_best - z) if maximize else (z - z_best)
    tail = ("<b>This is the optimal plan.</b>" if abs(gap) < 1e-3 * max(1, abs(z_best)) else
            f"The best feasible plan reaches {money(z_best)}: you are {money(abs(gap))} "
            f"{'below' if maximize else 'above'} it. Can you do better?")
    banner(True, f"<b>Feasible plan</b>: {what} = {money(z)}.<br>{tail}", top=True)


def slider(desc, value, lo, hi, step):
    return W.FloatSlider(value=value, min=lo, max=hi, step=step, description=desc, continuous_update=False,
                         style={"description_width": "150px"}, layout=W.Layout(width="420px"))


def explorer(groups, view, best, chart_beside_controls=False):
    """groups: {"Column title": {name: widget}}; best: {name: optimal value}. Redraws on every change.
    chart_beside_controls: the chart goes next to the controls (e.g. a map to look at while ticking roads)."""
    controls = {k: w for g in groups.values() for k, w in g.items()}
    cols = [W.VBox([W.HTML(f"<b>{title}</b>")] + list(g.values())) for title, g in groups.items()]
    out, chart = W.HTML(), W.HTML()

    def refresh(_=None):
        PARTS.clear()
        try:
            view(**{k: w.value for k, w in controls.items()})
        except Exception:
            plt.close("all")
            PARTS.append(f"<pre style='color:#bd0a0a'>{traceback.format_exc()}</pre>")
        # layout: verdict banner on top, then tables (left) and chart (right) side by side, so it fits one screen
        banners = [h for h in PARTS if h.startswith("<div")]
        figures = [h for h in PARTS if h.startswith("<img")]
        tables = [h for h in PARTS if h not in banners and h not in figures]
        figures = [h.replace("<img ", '<img style="width:100%;max-width:520px;height:auto" ', 1) for h in figures]
        if chart_beside_controls:
            chart.value = f'<div style="width:480px;max-width:100%">{"".join(figures)}</div>'
            out.value = "".join(banners) + "".join(tables)
            return
        out.value = ("".join(banners) +
                     '<div style="display:flex;flex-wrap:nowrap;gap:18px;align-items:flex-start">'
                     f'<div style="flex:0 0 auto">{"".join(tables)}</div>'
                     f'<div style="flex:1 1 280px;min-width:240px">{"".join(figures)}</div></div>')

    def load(values):
        for k, w in controls.items():
            w.unobserve(refresh, names="value")
            w.value = values(k)
            w.observe(refresh, names="value")
        refresh()

    b_opt = W.Button(description="Load the optimal plan", button_style="success", layout=W.Layout(width="200px"))
    b_zero = W.Button(description="Reset to 0", layout=W.Layout(width="120px"))
    b_opt.on_click(lambda _: load(lambda k: best[k]))
    b_zero.on_click(lambda _: load(lambda k: False if isinstance(controls[k], W.Checkbox) else 0.0))
    for w in controls.values():
        w.observe(refresh, names="value")
    refresh()
    top = W.HBox(cols + [chart]) if chart_beside_controls else W.HBox(cols)
    display(W.VBox([top, W.HBox([b_opt, b_zero]), out]))


_ANS = json.loads(base64.b64decode("e30="))


def check(key, value, tol=0.01):
    """Compare your answer with the expected one (relative tolerance 1%)."""
    if value is None:
        print(f"⏳ {key}: not answered yet"); return
    exp = _ANS[key]
    ok = abs(float(value) - exp) <= tol * max(1.0, abs(exp))
    print(f"✅ {key}: correct" if ok else f"❌ {key}: not yet")


print("Ready. Run each explorer cell below, then move the sliders.")

---
## 1 · The carpenter: what should she make this week?
A carpenter makes tables and chairs.

| | Profit (\$/unit) | Work (h/unit) | Material (units/unit) |
|---|---|---|---|
| Table | 500 | 2 | 1 |
| Chair | 300 | 1 | 2 |
| **Available each week** | | 40 | 50 |

**Try this**
* Make only tables, as many as possible. Which resource runs out? What is left of the other one?
* Make only chairs, as many as possible. Same questions.
* Find a better mix. On the graph, slide the dashed line (*same profit as your plan*) upward: where does it
  leave the blue region for the last time?

In [ ]:
# @title 🪑 Carpenter explorer (table profit $500): choose the plan, watch profit, hours and material
def _explorer():
    PROFIT = np.array([500, 300])                       # $ per table, per chair (fixed data)
    USE = np.array([[2, 1],                                # hours of work per table, per chair
                    [1, 2]])                               # units of material per table, per chair
    AVAILABLE = np.array([40, 50])
    RESOURCES = ["work (h)", "material (units)"]
    x_best, z_best = optimum(PROFIT, A_ub=USE, b_ub=AVAILABLE, bounds=[(0, None)] * 2, maximize=True)


    def carpenter_view(t, c):
        x = np.array([t, c])
        violated = constraints([(n, u, "≤", a) for n, u, a in zip(RESOURCES, USE @ x, AVAILABLE)])
        z = PROFIT @ x
        verdict(violated, z, z_best, "weekly profit", maximize=True)
        table(pd.DataFrame({"Product": ["Tables", "Chairs"], "Units": x, "Profit ($/unit)": PROFIT,
                            "Profit ($)": PROFIT * x}))
        fig, ax = plt.subplots(figsize=(4.6, 4.0))
        s = np.array([0, 30])
        ax.fill([0, 20, 10, 0], [0, 0, 20, 25], color="#c6dcff", alpha=0.6, label="feasible plans")
        ax.plot(s, 40 - 2 * s, color="#305bab", label="work: 2t + c = 40")
        ax.plot(s, (50 - s) / 2, color="#067429", label="material: t + 2c = 50")
        if z > 0:
            ax.plot(s, (z - PROFIT[0] * s) / PROFIT[1], "k--", lw=1, label="same profit as your plan")
        ax.scatter([t], [c], s=80, color="#bd0a0a" if violated else "#ff8a00", zorder=5, label="your plan")
        ax.set_xlim(0, 30); ax.set_ylim(0, 30); ax.set_xlabel("tables t"); ax.set_ylabel("chairs c")
        ax.legend(fontsize=7, loc="upper right")
        show(fig)


    explorer({"Your decisions: units made this week": dict(t=slider("t tables", 0, 0, 30, 0.5),
                                                            c=slider("c chairs", 0, 0, 30, 0.5))},
             carpenter_view, dict(zip(["t", "c"], x_best)))


_explorer()

### Formulation
**Variables:** $t$ = tables made, $c$ = chairs made this week.

$$
\begin{aligned}
\max\ & z = 500t + 300c && \text{(weekly profit)} \\
\text{s.t. } & 2t + c \le 40 && \text{(hours of work)} \\
& t + 2c \le 50 && \text{(units of material)} \\
& t, c \ge 0
\end{aligned}
$$

In [ ]:
import pulp

products = ["Table", "Chair"]
profit = {"Table": 500, "Chair": 300}               # $ per unit
work = {"Table": 2, "Chair": 1}                     # hours per unit
material = {"Table": 1, "Chair": 2}                 # units of material per unit
hours, wood = 40, 50                                # available each week

model = pulp.LpProblem("carpenter", pulp.LpMaximize)
x = {p: model.add_variable(f"x_{p}", lowBound=0) for p in products}   # units made
model += pulp.lpSum(profit[p] * x[p] for p in products)              # weekly profit
model += pulp.lpSum(work[p] * x[p] for p in products) <= hours, "work"
model += pulp.lpSum(material[p] * x[p] for p in products) <= wood, "material"
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· profit = ${pulp.value(model.objective):,.0f}")
for p in products:
    print(f"  {p:<6} {x[p].value():5.1f} units")

### Same question, new price
The market changes: a **table now earns only \$100** (chairs still earn \$300). Same workshop, same hours,
same material.

**Try this**
* Load your previous plan (10 tables, 20 chairs): is it still feasible? Is it still the best?
* Find the new best plan. Which resource is now left over?
* Compare the dashed line with the one of the first explorer: what changed, the region or the line?

In [ ]:
# @title 🪑 Carpenter explorer (table profit $100): choose the plan, watch profit, hours and material
def _explorer():
    PROFIT = np.array([100, 300])                       # $ per table, per chair (fixed data)
    USE = np.array([[2, 1],                                # hours of work per table, per chair
                    [1, 2]])                               # units of material per table, per chair
    AVAILABLE = np.array([40, 50])
    RESOURCES = ["work (h)", "material (units)"]
    x_best, z_best = optimum(PROFIT, A_ub=USE, b_ub=AVAILABLE, bounds=[(0, None)] * 2, maximize=True)


    def carpenter_view(t, c):
        x = np.array([t, c])
        violated = constraints([(n, u, "≤", a) for n, u, a in zip(RESOURCES, USE @ x, AVAILABLE)])
        z = PROFIT @ x
        verdict(violated, z, z_best, "weekly profit", maximize=True)
        table(pd.DataFrame({"Product": ["Tables", "Chairs"], "Units": x, "Profit ($/unit)": PROFIT,
                            "Profit ($)": PROFIT * x}))
        fig, ax = plt.subplots(figsize=(4.6, 4.0))
        s = np.array([0, 30])
        ax.fill([0, 20, 10, 0], [0, 0, 20, 25], color="#c6dcff", alpha=0.6, label="feasible plans")
        ax.plot(s, 40 - 2 * s, color="#305bab", label="work: 2t + c = 40")
        ax.plot(s, (50 - s) / 2, color="#067429", label="material: t + 2c = 50")
        if z > 0:
            ax.plot(s, (z - PROFIT[0] * s) / PROFIT[1], "k--", lw=1, label="same profit as your plan")
        ax.scatter([t], [c], s=80, color="#bd0a0a" if violated else "#ff8a00", zorder=5, label="your plan")
        ax.set_xlim(0, 30); ax.set_ylim(0, 30); ax.set_xlabel("tables t"); ax.set_ylabel("chairs c")
        ax.legend(fontsize=7, loc="upper right")
        show(fig)


    explorer({"Your decisions: units made this week": dict(t=slider("t tables", 0, 0, 30, 0.5),
                                                            c=slider("c chairs", 0, 0, 30, 0.5))},
             carpenter_view, dict(zip(["t", "c"], x_best)))


_explorer()

In [ ]:
import pulp

products = ["Table", "Chair"]
profit = {"Table": 100, "Chair": 300}               # the only change: the profit of a table
work = {"Table": 2, "Chair": 1}
material = {"Table": 1, "Chair": 2}
hours, wood = 40, 50

model = pulp.LpProblem("carpenter_new_price", pulp.LpMaximize)
x = {p: model.add_variable(f"x_{p}", lowBound=0) for p in products}
model += pulp.lpSum(profit[p] * x[p] for p in products)
model += pulp.lpSum(work[p] * x[p] for p in products) <= hours, "work"
model += pulp.lpSum(material[p] * x[p] for p in products) <= wood, "material"
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· profit = ${pulp.value(model.objective):,.0f}")
for p in products:
    print(f"  {p:<6} {x[p].value():5.1f} units")

---
## 2 · The workshop grows: six products, three resources, a market limit
| | Table | Chair | Desk | Bookcase | Bed | Cabinet | Available |
|---|---|---|---|---|---|---|---|
| Profit (\$/unit) | 500 | 300 | 700 | 450 | **900** | 600 | |
| Carpentry (h) | 1 | 5 | 4 | 2 | 5 | 2 | 160 h |
| Finishing (h) | 1 | 4 | 2 | 2 | 3 | 1 | 100 h |
| Wood (units) | 2 | 3 | 2 | 2 | 3 | 4 | 180 units |

At most **20 tables** can be sold per week.

**Try this**
* Make only the most profitable product (beds), as many as possible. What profit do you get?
* Try your own intuition for 2 minutes. What is your best profit?
* Load the optimal plan: which product is **not** made at all? Which resources are used to 100%?

In [ ]:
# @title 🏭 Workshop explorer: six products, three resources, trust your intuition…
def _explorer():
    PRODUCTS = ["Table", "Chair", "Desk", "Bookcase", "Bed", "Cabinet"]
    PROFIT = np.array([500, 300, 700, 450, 900, 600])      # $ per unit (fixed data)
    USE = np.array([[1, 5, 4, 2, 5, 2],                    # carpentry hours per unit
                    [1, 4, 2, 2, 3, 1],                    # finishing hours per unit
                    [2, 3, 2, 2, 3, 4]])                   # units of wood per unit
    AVAILABLE = np.array([160, 100, 180])
    RESOURCES = ["carpentry (h)", "finishing (h)", "wood (units)"]
    MAX_TABLES = 20                                        # market limit
    x_best, z_best = optimum(PROFIT, A_ub=USE, b_ub=AVAILABLE, bounds=[(0, MAX_TABLES)] + [(0, None)] * 5,
                             maximize=True)
    KEYS = [p.lower() for p in PRODUCTS]


    def workshop_view(**x):
        v = np.array([x[k] for k in KEYS])
        used = USE @ v
        rows = [(n, u, "≤", a) for n, u, a in zip(RESOURCES, used, AVAILABLE)]
        violated = constraints(rows + [("tables sold (market)", v[0], "≤", MAX_TABLES)])
        verdict(violated, PROFIT @ v, z_best, "weekly profit", maximize=True)
        table(pd.DataFrame({"Product": PRODUCTS, "Units": v, "Profit ($)": PROFIT * v}))
        fig, ax = plt.subplots(figsize=(5, 2.4))
        pct = 100 * used / AVAILABLE
        ax.barh(RESOURCES, pct, color=["#bd0a0a" if p > 100 + 1e-6 else "#659df2" for p in pct])
        ax.axvline(100, color="black", ls="--", lw=1)
        ax.set_xlabel("% of the resource used (dashed line: all of it)")
        ax.set_xlim(0, max(120, pct.max() * 1.1))
        show(fig)


    sliders = {k: slider(f"{p} (${c})", 0, 0, 40, 1) for k, p, c in zip(KEYS, PRODUCTS, PROFIT)}
    explorer({"Your decisions: units of each product": {k: sliders[k] for k in KEYS[:3]},
              "&nbsp;": {k: sliders[k] for k in KEYS[3:]}}, workshop_view, dict(zip(KEYS, x_best)))


_explorer()

### Formulation
**Variables:** $x_j$ = units of product $j$ made this week ($j$ = table, chair, desk, bookcase, bed, cabinet).

$$
\begin{aligned}
\max\ & z = 500x_T + 300x_C + 700x_D + 450x_K + 900x_B + 600x_A \\
\text{s.t. } & x_T + 5x_C + 4x_D + 2x_K + 5x_B + 2x_A \le 160 && \text{(carpentry)} \\
& x_T + 4x_C + 2x_D + 2x_K + 3x_B + x_A \le 100 && \text{(finishing)} \\
& 2x_T + 3x_C + 2x_D + 2x_K + 3x_B + 4x_A \le 180 && \text{(wood)} \\
& x_T \le 20 && \text{(market)} \\
& x_j \ge 0
\end{aligned}
$$

**General form:** $\max \sum_j p_j x_j$ s.t. $\sum_j a_{ij} x_j \le b_i$ for every resource $i$, $x_j \ge 0$:
the same model as the carpenter, with more products and more resources.

In [ ]:
import pulp

products = ["Table", "Chair", "Desk", "Bookcase", "Bed", "Cabinet"]
profit = dict(zip(products, [500, 300, 700, 450, 900, 600]))          # $ per unit
use = {"carpentry": dict(zip(products, [1, 5, 4, 2, 5, 2])),           # hours per unit
       "finishing": dict(zip(products, [1, 4, 2, 2, 3, 1])),           # hours per unit
       "wood": dict(zip(products, [2, 3, 2, 2, 3, 4]))}                # units per unit
available = {"carpentry": 160, "finishing": 100, "wood": 180}
max_tables = 20

model = pulp.LpProblem("workshop", pulp.LpMaximize)
x = {p: model.add_variable(f"x_{p}", lowBound=0) for p in products}
model += pulp.lpSum(profit[p] * x[p] for p in products)
for r in available:
    model += pulp.lpSum(use[r][p] * x[p] for p in products) <= available[r], r
model += x["Table"] <= max_tables, "market"
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· profit = ${pulp.value(model.objective):,.0f}")
for p in products:
    print(f"  {p:<9} {x[p].value():5.1f} units")

---
## 3 · Dorian Auto: advertise effectively but cheaply
Dorian Auto buys TV ads to reach at least **28 million women** and **24 million men**.

| Ad | Cost (\$/ad) | Women reached (millions) | Men reached (millions) |
|---|---|---|---|
| Comedy show | 50,000 | 7 | 2 |
| Football game | 100,000 | 2 | 12 |

This time we **minimize** a cost and the constraints are **at least** ($\ge$): the feasible region is
unbounded upward, and the solver pushes the cost line **down**.

**Try this**
* Buy only comedy ads, as few as possible. Which audience is reached last?
* Find the cheapest plan with the sliders. Is it a whole number of ads?
* Load the optimal plan, then round it to whole ads. Is the rounded plan still feasible? Look at the grey dots:
  which whole-number plan is the cheapest?

In [ ]:
# @title 📺 Dorian Auto explorer: choose the ads, watch cost and audience
def _explorer():
    COST = np.array([50_000, 100_000])                     # $ per comedy ad, per football ad (fixed data)
    REACH = np.array([[7, 2],                              # millions of women reached per ad
                      [2, 12]])                            # millions of men reached per ad
    NEED = np.array([28, 24])
    AUDIENCES = ["women (millions)", "men (millions)"]
    x_best, z_best = optimum(COST, A_ub=-REACH, b_ub=-NEED, bounds=[(0, None)] * 2)


    def dorian_view(x1, x2):
        x = np.array([x1, x2])
        violated = constraints([(n, r, "≥", d) for n, r, d in zip(AUDIENCES, REACH @ x, NEED)])
        verdict(violated, COST @ x, z_best, "advertising cost")
        table(pd.DataFrame({"Ad": ["Comedy", "Football"], "Ads bought": x, "Cost ($)": COST * x}))
        fig, ax = plt.subplots(figsize=(4.6, 4.0))
        s = np.array([0, 12])
        ax.fill([0, 3.6, 12, 12, 0], [14, 1.4, 0, 15, 15], color="#c6dcff", alpha=0.6, label="feasible plans")
        ax.plot(s, (28 - 7 * s) / 2, color="#305bab", label="women: 7x₁ + 2x₂ = 28")
        ax.plot(s, (24 - 2 * s) / 12, color="#067429", label="men: 2x₁ + 12x₂ = 24")
        gx, gy = np.meshgrid(np.arange(13), np.arange(16))
        ok = (7 * gx + 2 * gy >= 28) & (2 * gx + 12 * gy >= 24)
        ax.scatter(gx[ok], gy[ok], s=6, color="#595959", label="whole numbers of ads")
        if COST @ x > 0:
            ax.plot(s, (COST @ x - COST[0] * s) / COST[1], "k--", lw=1, label="same cost as your plan")
        ax.scatter([x1], [x2], s=80, color="#bd0a0a" if violated else "#ff8a00", zorder=5, label="your plan")
        ax.set_xlim(0, 12); ax.set_ylim(0, 15); ax.set_xlabel("comedy ads x₁"); ax.set_ylabel("football ads x₂")
        ax.legend(fontsize=7, loc="upper right")
        show(fig)


    explorer({"Your decisions: ads bought": dict(x1=slider("x₁ comedy ads", 0, 0, 12, 0.1),
                                                  x2=slider("x₂ football ads", 0, 0, 15, 0.1))},
             dorian_view, dict(zip(["x1", "x2"], x_best)))


_explorer()

### Formulation
**Variables:** $x_1$ = comedy ads, $x_2$ = football ads.

$$
\begin{aligned}
\min\ & z = 50{,}000x_1 + 100{,}000x_2 && \text{(cost)} \\
\text{s.t. } & 7x_1 + 2x_2 \ge 28 && \text{(women, millions)} \\
& 2x_1 + 12x_2 \ge 24 && \text{(men, millions)} \\
& x_1, x_2 \ge 0
\end{aligned}
$$

In [ ]:
import pulp

ads = ["Comedy", "Football"]
cost = {"Comedy": 50_000, "Football": 100_000}       # $ per ad
reach = {"women": {"Comedy": 7, "Football": 2},       # millions reached per ad
         "men": {"Comedy": 2, "Football": 12}}
need = {"women": 28, "men": 24}                       # millions, at least

model = pulp.LpProblem("dorian", pulp.LpMinimize)
x = {a: model.add_variable(f"x_{a}", lowBound=0) for a in ads}
model += pulp.lpSum(cost[a] * x[a] for a in ads)
for g in need:
    model += pulp.lpSum(reach[g][a] * x[a] for a in ads) >= need[g], g
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· cost = ${pulp.value(model.objective):,.0f}")
for a in ads:
    print(f"  {a:<8} {x[a].value():5.2f} ads")

### Making it an integer program
You cannot buy 3.6 ads. Adding **integrality** ($x_1, x_2$ whole numbers) drops the **divisibility**
assumption of LP; proportionality, additivity and certainty still hold. In PuLP, it is one word:
`cat="Integer"`. Compare the integer optimum with the rounded LP plan $(4, 1)$.

In [ ]:
import pulp

ads = ["Comedy", "Football"]
cost = {"Comedy": 50_000, "Football": 100_000}
reach = {"women": {"Comedy": 7, "Football": 2}, "men": {"Comedy": 2, "Football": 12}}
need = {"women": 28, "men": 24}

model = pulp.LpProblem("dorian_integer", pulp.LpMinimize)
x = {a: model.add_variable(f"x_{a}", lowBound=0, cat="Integer") for a in ads}   # whole ads only
model += pulp.lpSum(cost[a] * x[a] for a in ads)
for g in need:
    model += pulp.lpSum(reach[g][a] * x[a] for a in ads) >= need[g], g
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· cost = ${pulp.value(model.objective):,.0f}")
for a in ads:
    print(f"  {a:<8} {x[a].value():3.0f} ads")

rounded = {"Comedy": 4, "Football": 1}                # the LP optimum (3.6, 1.4), rounded
for g in need:
    got = sum(reach[g][a] * rounded[a] for a in ads)
    print(f"rounded plan (4, 1), {g}: {got} million {'≥' if got >= need[g] else '<'} {need[g]}")

---
## 4 · When the solver says no: infeasible and unbounded models
The solver does not always return a plan. Run the cell and read the **status** of each model.
* **Infeasible:** at least 10 units of A, at least 8 units of B, but at most 15 units in total.
* **Unbounded:** $\max\ 3x_1 + 2x_2$ s.t. $x_1 - x_2 \ge 1$: nothing limits the profit.

Both usually reveal a **modeling error**: a requirement that cannot be met, or a missing limit (resources,
market demand).

In [ ]:
import pulp

infeasible = pulp.LpProblem("production_gone_wrong", pulp.LpMaximize)
a = infeasible.add_variable("x_A", lowBound=0)
b = infeasible.add_variable("x_B", lowBound=0)
infeasible += a + b
infeasible += a >= 10, "at_least_10_A"
infeasible += b >= 8, "at_least_8_B"
infeasible += a + b <= 15, "at_most_15_in_total"
print("Infeasible example:", infeasible.solve(pulp.COIN_CMD(msg=False)).status_str)

unbounded = pulp.LpProblem("too_good_to_be_true", pulp.LpMaximize)
x1 = unbounded.add_variable("x1", lowBound=0)
x2 = unbounded.add_variable("x2", lowBound=0)
unbounded += 3 * x1 + 2 * x2
unbounded += x1 - x2 >= 1, "only_constraint"
print("Unbounded example: ", unbounded.solve(pulp.COIN_CMD(msg=False)).status_str)

---
## What did you learn?
* A model has three parts: **decision variables** (what you control), an **objective** (what you maximize or
  minimize) and **constraints** (what limits you). The **data** never change while you decide.
* With two variables, the feasible plans form a **region**, and the optimum sits at a **corner** of it: the
  corner where the profit (or cost) line leaves the region.
* Changing a price changes the **slope of the line**, not the region: the best corner can jump to another one.
* Intuition (the most profitable product first) can miss the optimum by a lot: the solver checks every resource
  at once.
* Whole-number decisions make the model an **integer program**; rounding the LP solution can be infeasible.